# Task 1 Evaluator / Viva Notebook ? Character-Level GPT

**Member:** Anshika Goel  
**Team:** 15

This notebook is a **read-only evaluation/demo notebook** for the
already-trained Task 1 character-level GPT.

It is intentionally designed for an evaluator or viva demonstration.

It will:

- locate the repository without a personal filesystem path;
- verify the frozen validation tensor, vocabulary, split manifest,
  checkpoint, and committed metric artifacts;
- load the saved `best_model.pt`;
- recompute validation cross-entropy, perplexity, bits per character,
  generalization gap, and top-1 next-character accuracy;
- compare recomputed values with the committed evaluation artifact;
- display the saved generation-diversity, gradient-stability,
  resource, and failure-analysis evidence.

It will **not** create an optimizer, call `backward()`, train,
update weights, overwrite checkpoints, or alter any experiment result.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import sys
import time
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
import yaml
from IPython.display import Markdown, display


def find_repo_root():
    current = Path.cwd().resolve()

    for candidate in [
        current,
        *current.parents,
    ]:
        expected = (
            candidate
            / "task1_llm"
            / "Anshika_Goel"
            / "configs"
            / "gpt_char.yaml"
        )

        if expected.is_file():
            return candidate

    raise RuntimeError(
        "Unable to locate repository root."
    )


def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


ROOT = find_repo_root()

MEMBER = (
    ROOT
    / "task1_llm"
    / "Anshika_Goel"
)

CONFIG_PATH = (
    MEMBER
    / "configs"
    / "gpt_char.yaml"
)

VOCABULARY_PATH = (
    MEMBER
    / "outputs"
    / "metrics"
    / "vocabulary.json"
)

SPLIT_MANIFEST_PATH = (
    MEMBER
    / "outputs"
    / "metrics"
    / "split_manifest.json"
)

EVALUATION_METRICS_PATH = (
    MEMBER
    / "outputs"
    / "metrics"
    / "evaluation_metrics.json"
)

TRAINING_SUMMARY_PATH = (
    MEMBER
    / "outputs"
    / "metrics"
    / "training_summary.json"
)

GENERATED_SAMPLES_PATH = (
    MEMBER
    / "outputs"
    / "samples"
    / "generated_samples.json"
)

VALIDATION_TENSOR_PATH = (
    ROOT
    / "task1_llm"
    / "data"
    / "processed"
    / "Anshika_Goel"
    / "validation_sequences.pt"
)

CHECKPOINT_PATH = (
    MEMBER
    / "checkpoints"
    / "best_model.pt"
)

FAILURE_ANALYSIS_PATH = (
    MEMBER
    / "failure_analysis.md"
)

EXPECTED_CHECKPOINT_SHA256 = (
    "64ad99efc0bf2e49959d3ef54bf5373"
    "a4c3a974f0fa333ae4a37e62fac9d2ba3"
)

print("Repository located.")
print(
    "Task 1 member folder:",
    MEMBER.relative_to(ROOT),
)


In [ ]:
required_paths = [
    CONFIG_PATH,
    VOCABULARY_PATH,
    SPLIT_MANIFEST_PATH,
    EVALUATION_METRICS_PATH,
    TRAINING_SUMMARY_PATH,
    GENERATED_SAMPLES_PATH,
    VALIDATION_TENSOR_PATH,
    CHECKPOINT_PATH,
    FAILURE_ANALYSIS_PATH,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(
            f"Required artifact missing: "
            f"{path.relative_to(ROOT)}"
        )


config = yaml.safe_load(
    CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)

vocabulary = json.loads(
    VOCABULARY_PATH.read_text(
        encoding="utf-8"
    )
)

split_manifest = json.loads(
    SPLIT_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

evaluation_metrics = json.loads(
    EVALUATION_METRICS_PATH.read_text(
        encoding="utf-8"
    )
)

training_summary = json.loads(
    TRAINING_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

generated_samples = json.loads(
    GENERATED_SAMPLES_PATH.read_text(
        encoding="utf-8"
    )
)


validation_sha = sha256_file(
    VALIDATION_TENSOR_PATH
)

vocabulary_sha = sha256_file(
    VOCABULARY_PATH
)

checkpoint_sha = sha256_file(
    CHECKPOINT_PATH
)


assert (
    validation_sha
    == str(
        split_manifest[
            "validation_tensor_sha256"
        ]
    ).lower()
)

assert (
    vocabulary_sha
    == str(
        split_manifest[
            "vocabulary_sha256"
        ]
    ).lower()
)

assert (
    checkpoint_sha
    == EXPECTED_CHECKPOINT_SHA256
)

assert (
    split_manifest[
        "story_overlap_between_splits"
    ]
    is False
)


provenance = pd.DataFrame(
    [
        {
            "Artifact": (
                "Validation tensor"
            ),
            "Repository path": str(
                VALIDATION_TENSOR_PATH
                .relative_to(ROOT)
            ),
            "SHA256 verified": True,
        },
        {
            "Artifact": "Vocabulary",
            "Repository path": str(
                VOCABULARY_PATH
                .relative_to(ROOT)
            ),
            "SHA256 verified": True,
        },
        {
            "Artifact": "Best checkpoint",
            "Repository path": str(
                CHECKPOINT_PATH
                .relative_to(ROOT)
            ),
            "SHA256 verified": True,
        },
    ]
)

display(provenance)

print(
    "Training sequences:",
    split_manifest[
        "train_sequences"
    ],
)

print(
    "Validation sequences:",
    split_manifest[
        "validation_sequences"
    ],
)

print(
    "Sequence length:",
    split_manifest[
        "sequence_length"
    ],
)

print(
    "Vocabulary size:",
    split_manifest[
        "vocabulary_size"
    ],
)

print(
    "Train/validation story overlap:",
    split_manifest[
        "story_overlap_between_splits"
    ],
)

print()
print(
    "ARTIFACT PROVENANCE CHECK: PASSED"
)


In [ ]:
code_directory = (
    MEMBER
    / "code"
)

if str(code_directory) not in sys.path:
    sys.path.insert(
        0,
        str(code_directory),
    )

from model import (
    build_model_from_experiment_config,
)


validation_tensor = torch.load(
    VALIDATION_TENSOR_PATH,
    map_location="cpu",
    weights_only=True,
)

assert isinstance(
    validation_tensor,
    torch.Tensor,
)

assert list(
    validation_tensor.shape
) == list(
    split_manifest[
        "validation_tensor_shape"
    ]
)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

vocab_size = int(
    vocabulary["vocab_size"]
)

model = (
    build_model_from_experiment_config(
        config,
        vocab_size,
    )
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True,
)


if (
    isinstance(checkpoint, dict)
    and "model_state_dict"
    in checkpoint
):
    state_dict = checkpoint[
        "model_state_dict"
    ]

elif (
    isinstance(checkpoint, dict)
    and "state_dict"
    in checkpoint
):
    state_dict = checkpoint[
        "state_dict"
    ]

elif (
    isinstance(checkpoint, dict)
    and checkpoint
    and all(
        torch.is_tensor(value)
        for value
        in checkpoint.values()
    )
):
    state_dict = checkpoint

else:
    raise RuntimeError(
        "Unable to identify model state "
        "inside Task 1 checkpoint."
    )


model.load_state_dict(
    state_dict
)

model.to(device)
model.eval()


print(
    "Evaluation device:",
    device,
)

if device.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

print(
    "Parameter count:",
    sum(
        parameter.numel()
        for parameter
        in model.parameters()
    ),
)

print(
    "Checkpoint loaded for evaluation only."
)


In [ ]:
batch_size = 32

loss_sum = 0.0
correct_count = 0
token_count = 0

start_time = time.perf_counter()


with torch.inference_mode():

    for start in range(
        0,
        validation_tensor.shape[0],
        batch_size,
    ):

        rows = (
            validation_tensor[
                start:
                start + batch_size
            ]
            .long()
            .to(device)
        )

        input_ids = rows[:, :-1]
        targets = rows[:, 1:]


        output = model(
            input_ids,
            targets=None,
            return_attentions=False,
        )


        if torch.is_tensor(output):
            logits = output

        elif isinstance(
            output,
            (tuple, list),
        ):
            logits = output[0]

        elif isinstance(
            output,
            dict,
        ):
            logits = output[
                "logits"
            ]

        else:
            raise RuntimeError(
                "Unexpected model forward "
                "output type."
            )


        batch_loss_sum = (
            F.cross_entropy(
                logits.reshape(
                    -1,
                    logits.shape[-1],
                ),
                targets.reshape(-1),
                reduction="sum",
            )
        )

        loss_sum += float(
            batch_loss_sum.item()
        )

        predictions = (
            logits.argmax(
                dim=-1
            )
        )

        correct_count += int(
            (
                predictions
                == targets
            )
            .sum()
            .item()
        )

        token_count += int(
            targets.numel()
        )


elapsed = (
    time.perf_counter()
    - start_time
)

validation_ce = (
    loss_sum
    / token_count
)

perplexity = math.exp(
    validation_ce
)

bits_per_character = (
    validation_ce
    / math.log(2.0)
)

top1_accuracy = (
    correct_count
    / token_count
)

final_training_loss = float(
    evaluation_metrics[
        "training"
    ][
        "final_training_loss"
    ]
)

generalization_gap = (
    validation_ce
    - final_training_loss
)


recomputed = {
    "cross_entropy_loss":
        validation_ce,

    "perplexity":
        perplexity,

    "bits_per_character":
        bits_per_character,

    "generalization_gap":
        generalization_gap,

    "top1_accuracy":
        top1_accuracy,
}


saved = (
    evaluation_metrics[
        "validation"
    ]
)


rows = []

tolerances = {
    "cross_entropy_loss": 5e-4,
    "perplexity": 1e-3,
    "bits_per_character": 1e-3,
    "generalization_gap": 5e-4,
    "top1_accuracy": 1e-4,
}


for metric_name in [
    "cross_entropy_loss",
    "perplexity",
    "bits_per_character",
    "generalization_gap",
    "top1_accuracy",
]:

    difference = abs(
        float(
            recomputed[
                metric_name
            ]
        )
        - float(
            saved[
                metric_name
            ]
        )
    )

    matches = (
        difference
        <= tolerances[
            metric_name
        ]
    )

    rows.append(
        {
            "Metric": metric_name,
            "Recomputed": (
                recomputed[
                    metric_name
                ]
            ),
            "Committed": (
                saved[
                    metric_name
                ]
            ),
            "Absolute difference":
                difference,
            "Within tolerance":
                matches,
        }
    )

    if not matches:
        raise RuntimeError(
            "Task 1 metric mismatch: "
            + metric_name
        )


display(
    pd.DataFrame(rows)
)

print(
    "Validation tokens:",
    token_count,
)

print(
    "Evaluator wall seconds:",
    elapsed,
)

print()
print(
    "TASK 1 VALIDATION RECOMPUTATION: PASSED"
)


In [ ]:
diversity = (
    evaluation_metrics[
        "diversity"
    ][
        "all_sampled_outputs"
    ]
)

training = (
    evaluation_metrics[
        "training"
    ]
)

generation = (
    evaluation_metrics[
        "generation"
    ]
)


diversity_table = pd.DataFrame(
    [
        {
            "Metric": "Distinct-1",
            "Committed value":
                diversity[
                    "distinct_1"
                ],
        },
        {
            "Metric": "Distinct-2",
            "Committed value":
                diversity[
                    "distinct_2"
                ],
        },
        {
            "Metric": "Distinct-3",
            "Committed value":
                diversity[
                    "distinct_3"
                ],
        },
        {
            "Metric":
                "Repeated 4-gram rate",
            "Committed value":
                diversity[
                    "repeated_4gram_rate"
                ],
        },
    ]
)

display(
    Markdown(
        "## Saved generation-diversity evidence"
    )
)

display(diversity_table)


stability_table = pd.DataFrame(
    [
        {
            "Metric":
                "Mean gradient norm",
            "Committed value":
                training[
                    "gradient_norm_mean"
                ],
        },
        {
            "Metric":
                "Maximum gradient norm",
            "Committed value":
                training[
                    "gradient_norm_max"
                ],
        },
        {
            "Metric":
                "Loss spikes",
            "Committed value":
                training[
                    "loss_spike_count"
                ],
        },
        {
            "Metric":
                "Nonfinite losses",
            "Committed value":
                training[
                    "nonfinite_loss_count"
                ],
        },
        {
            "Metric":
                "Nonfinite-gradient AMP skips",
            "Committed value":
                training[
                    "nonfinite_gradient_count"
                ],
        },
    ]
)

display(
    Markdown(
        "## Saved gradient-stability evidence"
    )
)

display(stability_table)


resource_table = pd.DataFrame(
    [
        {
            "Metric":
                "Training characters/second",
            "Committed value":
                training[
                    "training_tokens_per_second"
                ],
        },
        {
            "Metric":
                "Generation characters/second",
            "Committed value":
                generation[
                    "tokens_per_second"
                ],
        },
        {
            "Metric":
                "Training peak GPU allocated MB",
            "Committed value":
                training[
                    "peak_gpu_allocated_mb"
                ],
        },
        {
            "Metric":
                "Training peak GPU reserved MB",
            "Committed value":
                training[
                    "peak_gpu_reserved_mb"
                ],
        },
        {
            "Metric":
                "Total training seconds",
            "Committed value":
                training[
                    "total_training_seconds"
                ],
        },
    ]
)

display(
    Markdown(
        "## Saved resource evidence"
    )
)

display(resource_table)

print(
    "These resource/stability values are "
    "historical measurements from the "
    "original training/evaluation run; "
    "they are not regenerated by this notebook."
)


In [ ]:
display(
    Markdown(
        "## Task 1 failure analysis"
    )
)

display(
    Markdown(
        FAILURE_ANALYSIS_PATH.read_text(
            encoding="utf-8"
        )
    )
)


## Task 1 evaluator boundary

A successful run of this notebook demonstrates that the saved
Task 1 model and frozen validation tensor reproduce the reported
validation metrics within explicit numerical tolerances.

Training-time throughput, GPU memory, gradient statistics, and
previously generated-text diversity are historical evidence from
the original run and are displayed from committed artifacts rather
than recreated through retraining.
